# EE 243 — Assignment 3, Problem 2

**Fundamental matrix estimation (3 pts)**.

Pipeline: corners → ResNet50 on **8×8** patches → matching → 8-point algorithm with random subsets → lowest mean Sampson error.

> Do **not** use built-in fundamental matrix estimators (`findFundamentalMat`, etc.).

In [ ]:
# Setup
# !pip install opencv-python numpy matplotlib torch torchvision

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision.models import resnet50, ResNet50_Weights
from pathlib import Path

DATA_DIR = Path('.')
LEFT_PATH = DATA_DIR / 'viprectification_deskLeft.png'
RIGHT_PATH = DATA_DIR / 'viprectification_deskRight.png'

# assert LEFT_PATH.exists() and RIGHT_PATH.exists()


In [ ]:
def detect_corners(gray, max_corners=500):
    """Detect corner keypoints. Returns (N, 2) float array (x, y)."""
    pts = cv2.goodFeaturesToTrack(gray, maxCorners=max_corners, qualityLevel=0.01, minDistance=8)
    return pts.reshape(-1, 2).astype(np.float64)


def extract_patch_features(image_bgr, keypoints, patch_size=8, model=None, transform=None, device='cpu'):
    """ResNet50 features from patch_size regions around each keypoint."""
    # TODO: crop/pad 8x8 patches, resize for ResNet if needed, forward through model
    raise NotImplementedError


def match_features(feat_left, feat_right, ratio_thresh=0.75):
    """Return (M, 2) index pairs (left idx, right idx)."""
    raise NotImplementedError


def normalize_points(pts):
    centroid = pts.mean(axis=0)
    d = np.sqrt(((pts - centroid) ** 2).sum(axis=1)).mean()
    s = np.sqrt(2) / (d + 1e-8)
    T = np.array([
        [s, 0, -s * centroid[0]],
        [0, s, -s * centroid[1]],
        [0, 0, 1],
    ])
    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    pts_n = (T @ pts_h.T).T[:, :2]
    return T, pts_n


def eight_point_fundamental(pts1, pts2):
    """Estimate rank-2 fundamental matrix F. No cv2.findFundamentalMat."""
    raise NotImplementedError


def sampson_distance(F, pts1, pts2):
    pts1_h = np.hstack([pts1, np.ones((len(pts1), 1))])
    pts2_h = np.hstack([pts2, np.ones((len(pts2), 1))])
    Fx1 = (F @ pts1_h.T).T
    Ftx2 = (F.T @ pts2_h.T).T
    num = np.sum(pts2_h * Fx1, axis=1) ** 2
    den = Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2
    return (num / (den + 1e-12))


def ransac_fundamental(pts1, pts2, n_iter=2000, sample_size=8):
    """Return best F and mean Sampson error on all matches."""
    raise NotImplementedError


In [ ]:
img_l = cv2.imread(str(LEFT_PATH))
img_r = cv2.imread(str(RIGHT_PATH))
gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

kp_l = detect_corners(gray_l)
kp_r = detect_corners(gray_r)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).to(device).eval()
model.fc = torch.nn.Identity()
transform = weights.transforms()

feat_l, kp_l = extract_patch_features(img_l, kp_l, patch_size=8, model=model, transform=transform, device=device)
feat_r, kp_r = extract_patch_features(img_r, kp_r, patch_size=8, model=model, transform=transform, device=device)

matches = match_features(feat_l, feat_r)
pts_l = kp_l[matches[:, 0]]
pts_r = kp_r[matches[:, 1]]
print(f'Matched points: {len(pts_l)}')

F_best, mean_err = ransac_fundamental(pts_l, pts_r, n_iter=2000)
print('Best fundamental matrix F:')
print(F_best)
print(f'Mean Sampson error: {mean_err:.6f}')
